In [11]:
import pickle
import torch
import numpy as np
from terrarium.dataloaders import get_dataloader

### Demo

In [2]:
with open('./data/reviews/v_32768/tok.pkl','rb') as f:
    tok = pickle.load(f)

In [24]:
sos = tok.stoi["<|sos|>"]   # 257
sotext = tok.stoi["<|sotext|>"]   # 257

In [ ]:
B=1
T=512
dl = get_dataloader('targetguided',B,T,'./data/reviews/v_32768/train_token_ids.bin','./data/reviews/v_32768/train_target_ids.bin',"cpu")

In [ ]:
x,y,r = dl.next_batch()

In [ ]:
x = x.squeeze(0)
x

In [ ]:
sos_locs = np.where(x == sos)[0]
sos_locs

In [ ]:
# get list of ratings
r = r.squeeze(0)
ratings = r[sos_locs].tolist()
ratings

In [ ]:
reviews = [x[sos_locs[i]:sos_locs[i + 1]].tolist() for i in range(len(sos_locs) - 1)] # split on sos
reviews.append(x[sos_locs[-1]:].tolist()) # last review
reviews = [r[:-1] for r in reviews] # remove rating
print(reviews)

In [ ]:
# for viewing decoded reviews
decoded = [tok.decode(r) for r in reviews]
print(decoded)

In [ ]:
# for gpt-oss format
decoded = []
for rtoks in reviews:
    decoded.append(tok.decode(rtoks))

In [ ]:
decoded

### Full Code

In [ ]:
B=1
T=512
dl = get_dataloader('targetguided',B,T,'./data/reviews/v_32768/train_token_ids.bin','./data/reviews/v_32768/train_target_ids.bin',"cpu")

MAX_REVIEWS = 1000
all_reviews = []
all_ratings = []

while len(all_reviews) < MAX_REVIEWS:
    x, y, r = dl.next_batch()
    x = x.squeeze(0)
    r = r.squeeze(0)

    sos_locs = np.where(x == sos)[0]

    r = r.squeeze(0)
    ratings = r[sos_locs].tolist() # get first rating
    all_ratings.extend(ratings[:-1]) # drop last review bc likely partial

    reviews = [x[sos_locs[i]:sos_locs[i + 1]].tolist() for i in range(len(sos_locs) - 1)] # split on sos, drop last review bc likely partial
    reviews = [rev[:-1] for rev in reviews]  # remove rating
    all_reviews.extend(reviews)

all_reviews = all_reviews[:MAX_REVIEWS]
all_ratings = all_ratings[:MAX_REVIEWS]

decoded_reviews = []
for rtoks in all_reviews:
    decoded_reviews.append(tok.decode(rtoks))

In [ ]:
len(decoded_reviews), decoded_reviews

In [ ]:
print(all_ratings)

### Full Star Rating Code

In [19]:
B = 1
T = 512
dl = get_dataloader('targetguided',B,T,'./data/reviews/v_32768/train_token_ids.bin','./data/reviews/v_32768/train_target_ids.bin',"cpu")

TARGET_RATING = 3
MAX_REVIEWS = 300

three_star_reviews = []
three_star_ratings = []

while len(three_star_reviews) < MAX_REVIEWS:
    x, y, r = dl.next_batch()
    x = x.squeeze(0)
    r = r.squeeze(0)

    sos_locs = np.where(x == sos)[0]

    # ratings aligned with complete reviews (drop last)
    ratings = r[sos_locs[:-1]].tolist()

    # complete reviews only
    reviews = [
        x[sos_locs[i]:sos_locs[i + 1]].tolist()
        for i in range(len(sos_locs) - 1)
    ]
    reviews = [rev[:-1] for rev in reviews]  # remove rating token

    # filter for 3-star
    for rev, rating in zip(reviews, ratings):
        if rating == TARGET_RATING:
            three_star_reviews.append(rev)
            three_star_ratings.append(rating)

            if len(three_star_reviews) >= MAX_REVIEWS:
                break

In [20]:
decoded_reviews = []
for rtoks in three_star_reviews:
    decoded_reviews.append(tok.decode(rtoks))

In [21]:
len(decoded_reviews), decoded_reviews

(300,
 ["<|sos|><|household|><|sotitle|>the fabric color didn't bleed much, but it has ...<|sotext|>the fabric color didn't bleed much, but it has a LOT of scent in it that gets into your fabric.<|sor|>",
  "<|sos|><|books|><|sotitle|>More basic than practical.<|sotext|>The book is easy to read and covers the basics. It however, doesn't offer any insight in how to put the pieces together to build programs.<|sor|>",
  '<|sos|><|electronics|><|sotitle|>Almost perfect carrying case for the Zune player<|sotext|>The product was delivered in a timely manner and in the condition it was described.  In this case it was new and in the original packaging.<br />I was looking for a case to protect my Zune player and this was perfect, well almost perfect.  This would have been the perfect case if it had a belt clip.  Maybe they will catch it on the next design.<|sor|>',
  "<|sos|><|patio|><|sotitle|>Hard read<|sotext|>Not sure, can't seem to get into it.  I have tried to read this book because I lov

In [22]:
print(three_star_ratings)

[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


In [25]:
half_reviews = []

for rtoks in three_star_reviews:
    # find <|sotext|>
    try:
        idx = rtoks.index(sotext)
    except ValueError:
        continue  # safety: skip malformed reviews

    prefix = rtoks[:idx + 1]        # include <|sotext|>
    body   = rtoks[idx + 1:]        # tokens after <|sotext|>

    half = len(body) // 2
    truncated = prefix + body[:half]

    half_reviews.append(truncated)

In [28]:
decoded_reviews = []
for rtoks in half_reviews:
    decoded_reviews.append(tok.decode(rtoks))

In [29]:
len(decoded_reviews), decoded_reviews

(300,
 ["<|sos|><|household|><|sotitle|>the fabric color didn't bleed much, but it has ...<|sotext|>the fabric color didn't bleed much, but it has a",
  '<|sos|><|books|><|sotitle|>More basic than practical.<|sotext|>The book is easy to read and covers the basics. It however, doesn',
  '<|sos|><|electronics|><|sotitle|>Almost perfect carrying case for the Zune player<|sotext|>The product was delivered in a timely manner and in the condition it was described.  In this case it was new and in the original packaging.<br />I was looking for a case to protect',
  "<|sos|><|patio|><|sotitle|>Hard read<|sotext|>Not sure, can't seem to get into it.  I have tried to read this book because I",
  '<|sos|><|sports|><|sotitle|>Lacks build quality!<|sotext|>I have mixed emotions about this product.  The headlight quality seems to be pretty good, but the tail light seems to be very bad quality material.  The plastic seems very cheap, and when you open it to install the batteries, you can feel how chea